# Предобработка данных

### Набор данных состоит из различных проектов и генерируется из приложений для управления проектами, которые используются для обеспечения качества, безопасности и управления строительными площадками. В наборе данных есть два файла, а именно формы PM-данных о строительстве и задачи PM-данных о строительстве: -

-  Необходимо предварительно ознакомиться с данными. Проведём первичное знакомство со структурой данных, качеством данных. Удалим ненужные столбцы, откорректируем типы данных, обработаем пропущенные значения.

### После подготовки данных, была построена панель анализа данных. Доступна по ссылке ниже


<br>
<div style='background:#E1E9FF; padding: 12px'>
<b></b> <a style='color:blue; font-size:22px'href='https://datalens.yandex/r4ncefl5lp7id'>DataLens</a>
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")      

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
forms_df = pd.read_csv('Construction_Data_PM_Forms_All_Projects.csv')
tasks_df = pd.read_csv('Construction_Data_PM_Tasks_All_Projects.csv')

## Таблица: Forms
- **Ref**: Ссылка на элемент формы
- **Status**: Статус действия
- **Location**: Местоположение элемента контрольного списка (например, приложение для поля/локация на объекте)
- **Name**: Название формы
- **Created**: Дата создания формы
- **Type**: Источник формы (например, проверки управления объектом/проверки безопасности/разрешения/другое)
- **Status Changed**: Дата завершения задачи
- **Open Actions**: Неизвестно
- **Total Actions**: Неизвестно
- **Association**: Неизвестно
- **Overdue**: Просрочено ли действие формы (например, True/False)
- **Images**: Связанные изображения для формы
- **Comments**: Связанные комментарии для формы
- **Documents**: Связанные документы для формы
- **Project**: ID проекта
- **Report Forms Status**: Статус отчетной формы (например, open/closed)
- **Report Forms Group**: Группа, связанная с формой (например, Качество/Управление объектом/Другое)

## Таблица: Tasks
- **Ref**: Ссылка на элемент задачи
- **Status**: Статус действия (например, open/closed/EHS good observation/other)
- **Location**: Откуда взята задача (например, управление проверками EHS/форма EHS/другое)
- **Description**: Описание задачи управления проектом
- **Created**: Дата создания элемента
- **Target**: Неизвестно
- **Type**: Система светофора (например, Уведомление о безопасности Amber/Green/Other)
- **To Package**: Пакет, необходимый для выполнения задачи (например, генеральный подрядчик/железобетон/потолки и перегородки и т. д.)
- **Status Changed**: Дата завершения задачи
- **Association**: Неизвестно
- **Overdue**: Просрочено ли действие задачи (например, True/False)
- **Images**: Связанные изображения для задачи
- **Comments**: Связанные комментарии для задачи
- **Documents**: Связанные документы для задачи
- **Priority**: Приоритет задачи
- **Cause**: Причина создания элемента задачи (например, уборка/доступ к безопасности/другое)
- **Project**: ID проекта
- **Report Status**: Статус отчетной задачи (например, open/closed)
- **Task Group**: Группа, связанная с задачей (например, Безопасность/Управление объектом/Другое)

In [4]:
tasks_df.sample(3)

,Ref,Status,Location,Description,Created,Target,Type,To Package,Status Changed,Association,OverDue,Images,Comments,Documents,Priority,Cause,project,Report Status,Task Group
5954,T86424.468,EHS Good Observation,JPC Project Management>EHS Forms>01 Inspections,L1 - Spotters with MEWP,22/05/2020,NaN,Safety Notice (Green) - Good Observation,Painting,22/05/2020,FormAnswer,False,True,False,False,NaN,JPC - Safety - Plant & People Interface,1330,Closed,Safety
12415,T122039.13,Closed,1345 - DUB062 Project,Add more signage to entrance,11/03/2020,NaN,Safety Notice (Amber) - General Issue,Main Contractor,12/05/2020,FormAnswer,False,True,False,False,System Failure,JPC - Safety - Public Interface / Hoarding,1345,Closed,Safety
6597,T109023.77,EHS Good Observation,JPC Project Management>EHS Forms>01 Inspections,huge improvement on implementation of new rule...,07/04/2020,NaN,Safety Notice (Green) - Good Observation,Internal Partitions & Ceilings,07/04/2020,FormAnswer,False,False,False,False,NaN,JPC - Safety - Welfare Facilities,1330,Closed,Safety


In [5]:
forms_df.sample(3)

,Ref,Status,Location,Name,Created,Type,Status Changed,Open Actions,Total Actions,Association,OverDue,Images,Comments,Documents,Project,Report Forms Status,Report Forms Group
686,F1.469349,JPC Signed Off / Closed,02 Daily Work Plan>Site Management>JPC Project...,SM-FRM-SUB-101 Daily Work Plan,17/08/2020,Subcontractor Inspections,08/09/2020,0,0,NaN,False,False,False,False,1328,Closed,Subcontractor
9965,F129826.56,Closed,Block 5>ITP 01 Structural>Quality Control & BC...,QM-FRM-QC-010 RFIT - Request for Inspection or...,17/08/2020,Quality 00 General,19/08/2020,0,0,NaN,False,True,True,False,1345,Closed,Quality
7050,F109023.47,Closed,JPC Project Management,SM-FRM-001 Daily Site Diary,08/01/2020,Site Management,24/04/2020,0,0,NaN,False,True,True,False,1330,Closed,Site Management


In [6]:
forms_df["Created"] = pd.to_datetime(forms_df["Created"], format="%d/%m/%Y").dt.date
forms_df["Status Changed"] = pd.to_datetime(forms_df["Status Changed"], format="%d/%m/%Y").dt.date

In [7]:
# Указываем индексы столбцов, которые нужно обработать
selected_columns = [1, 2, 3, 5, -2, -1]  

# Цикл для обработки только указанных столбцов
for col_index in selected_columns:
    column_name = forms_df.columns[col_index] 
    col_data = forms_df[column_name]
    unique_values = col_data.unique()
    if len(unique_values) < 30:
        print(f"Cтолбец: {column_name}")
        print("Уникальные значения:", unique_values)
        print("Количество уникальных значений:", len(unique_values))
    else:
        print(f"Cтолбец: {column_name}")
        print("Количество уникальных значений:", len(unique_values))

Cтолбец: Status
Уникальные значения: ['Opened' 'Open / Ongoing Works' 'Subcontractor Signed Off'
 'Closed JPC Reviewed' 'JPC Signed Off / Closed' 'Closed'
 'JPC Sign Off / Closed' 'Rejected / Action Required' 'JPC Sign Off'
 'Works Complete / Resolved' 'Closed JPC Sign Off'
 'Completed and signed off' 'Open' 'In Process / Manufactor'
 'JPC Inspected' '3rd Party Sign Off - Closed' 'Ready to be delivered'
 'In Manufactor' 'Notification' 'Test Results Required' 'Works Complete'
 'Delivered to site' 'Permit to Unload / Dismantle (Closed)'
 'Permit to Load Signed Off' 'Closed (Design Team Acceptance)'
 'Ready to Inspect']
Количество уникальных значений: 26
Cтолбец: Location
Количество уникальных значений: 814
Cтолбец: Name
Количество уникальных значений: 125
Cтолбец: Type
Уникальные значения: ['Site Management' 'Subcontractor Inspections' 'Quality 00 General'
 'Permits' 'Safety Forms' 'Quality 02 Architectural'
 'Quality 04 MEP Services' 'Quality 01 Structural' '00 Project Management'
 'Des

In [8]:
tasks_df = pd.read_csv('Construction_Data_PM_Tasks_All_Projects.csv')

In [9]:
tasks_df.sample(3)

,Ref,Status,Location,Description,Created,Target,Type,To Package,Status Changed,Association,OverDue,Images,Comments,Documents,Priority,Cause,project,Report Status,Task Group
8789,T102906.7,Closed,JPC Project Management>EHS Management>EHS Insp...,straight ladders to be removed to storage area,18/03/2020,NaN,Safety Notice (Amber) - General Issue,Scaffolding Package,19/03/2020,NaN,False,True,False,False,NaN,JPC - Safety - Storage,1335,Closed,Safety
11405,T122032.78,EHS Good Observation,JPC Project Management>EHS Inspections,Good quality DSS,27/03/2020,NaN,Safety Notice (Green) - Good Observation,Demolition,27/03/2020,FormAnswer,False,True,False,False,NaN,JPC - Safety - Documentation,1340,Closed,Safety
3868,T133598.48,EHS Good Observation,JPC Project Management>EHS Management>01 Inspe...,Temporary staircase,21/08/2020,NaN,Safety Notice (Green) - Good Observation,Scaffolding,21/08/2020,FormAnswer,False,True,False,False,NaN,JPC - Safety - Access,1329,Closed,Safety


In [10]:
tasks_df["Created"] = pd.to_datetime(tasks_df["Created"], format="%d/%m/%Y").dt.date
tasks_df["Status Changed"] = pd.to_datetime(tasks_df["Status Changed"], format="%d/%m/%Y").dt.date

In [11]:
# Указываем индексы столбцов, которые нужно обработать
selected_columns = [0, 1,2,3,5, 6, 7, 9, -1, -2, -4, -5]  

# Цикл для обработки только указанных столбцов
for col_index in selected_columns:
    column_name = tasks_df.columns[col_index] 
    col_data = tasks_df[column_name]
    unique_values = col_data.unique()
    if len(unique_values) < 30:
        print(f"Cтолбец: {column_name}")
        print("Уникальные значения:", unique_values)
        print("Количество уникальных значений:", len(unique_values))
    else:
        print(f"Cтолбец: {column_name}")
        print("Количество уникальных значений:", len(unique_values))

Cтолбец: Ref
Количество уникальных значений: 12118
Cтолбец: Status
Уникальные значения: ['Open' 'Closed' 'EHS Good Observation' 'Open / Ongoing Works' 'Complete'
 'Works Complete / Resolved' '3rd Party Sign Off - Closed'
 'JPC Sign Off / Closed' 'JPC Sign Off' 'Rejected / Action Required'
 'Closed JPC Reviewed' '3rd Party Inspection - Closed' 'Photo Record'
 'JPC Inspected' 'Delivered - Material onsite' 'JPC Signed Off / Closed']
Количество уникальных значений: 16
Cтолбец: Location
Количество уникальных значений: 594
Cтолбец: Description
Количество уникальных значений: 10268
Cтолбец: Target
Количество уникальных значений: 256
Cтолбец: Type
Количество уникальных значений: 37
Cтолбец: To Package
Количество уникальных значений: 106
Cтолбец: Association
Уникальные значения: ['FormAnswer' nan 'ForwardedFrom' 'ForwardedTo']
Количество уникальных значений: 4
Cтолбец: Task Group
Уникальные значения: ['Safety' 'Site Management' 'Quality' 'Design Team' nan]
Количество уникальных значений: 5
Cтол

In [12]:
tasks_df = tasks_df.rename(columns={'project': 'Project'})

In [13]:
# оставляем столбцы:
columns_forms = ['Status', 'OverDue', 'Type', 'Report Forms Group', 'Created', 'Project', 'Ref', 'Name', 'Report Forms Status']
columns_tasks = ['Status', 'OverDue', 'Priority', 'Cause', 'Created', 'Project', 'Ref', 'Description', 'Report Status', 'Task Group']

In [14]:
# Удаление ненужных столбцов
filtered_forms = forms_df[columns_forms]
filtered_tasks = tasks_df[columns_tasks]

In [15]:
# получим строки с отсутствующими значениями по задачам
filtered_tasks[filtered_tasks.isnull().any(axis=1)]

,Status,OverDue,Priority,Cause,Created,Project,Ref,Description,Report Status,Task Group
1,Closed,False,NaN,NaN,2020-09-14,1328,T116412.200,Metsec,Closed,Site Management
2,EHS Good Observation,False,NaN,JPC - Safety - Access,2020-09-14,1328,T141663.27,Good clear exclusion zones and access through ...,Closed,Safety
3,Closed,False,NaN,NaN,2020-09-14,1328,T116412.199,RC walls,Closed,Site Management
4,EHS Good Observation,False,NaN,JPC - Safety - House Keeping,2020-09-14,1328,T141663.26,"block 02 working level has good housekeeping, ...",Closed,Safety
5,Closed,False,NaN,NaN,2020-09-14,1328,T116412.198,A3 Roofing,Closed,Site Management
...,...,...,...,...,...,...,...,...,...,...
12416,EHS Good Observation,False,NaN,JPC - Safety - Exclusion Zones,2020-03-11,1345,T122039.11,Good management of exclusion zones to overhead...,Closed,Safety
12418,EHS Good Observation,False,NaN,JPC - Safety - Exclusion Zones,2020-03-11,1345,T122039.7,Exclusion zone to overhead lines well maintained,Closed,Safety
12420,EHS Good Observation,False,NaN,JPC - Safety - Welfare Facilities,2020-03-11,1345,T122039.3,Canteen in condition,Closed,Safety
12422,Closed,False,NaN,JPC - Safety - Plant & Equipment,2020-02-20,1345,T120669.2,ga1 not available for some plant on site. requ...,Closed,Safety


In [16]:
# проверим столбец - Priority
filtered_tasks['Priority'].unique()

array(['Behavioural Failure', nan, 'System Failure', 'High', 'Medium',
       'Low', 'Medium (resolve within 5 days)',
       'High (resolve within 48 hours)', 'Low (resolve within 2 weeks)',
       '.', 'Best Practice',
       'System Failure - Deviation from RAMS / Manufacturer Instructions',
       '1 Month Look Ahead', '2 Week Look Ahead', '1 Week Look Ahead'],
      dtype=object)

In [17]:
filtered_tasks['Cause'].unique()

array(['JPC - Safety - Documentation', nan, 'JPC - Safety - Access',
       'JPC - Safety - House Keeping',
       'JPC - Safety - Welfare Facilities', 'JPC - Safety - PPE',
       'JPC - Safety - Storage', 'JPC - Safety - Ladders',
       'JPC - Safety - WAH Equipment', 'JPC - Safety - Scaffolding',
       'JPC - Quality - Workmanship',
       'JPC - Safety - Temporary Electrics', 'JPC - Safety - Electrical',
       'JPC - Safety - Lead Management',
       'JPC - Safety - Working At Height',
       'JPC - Program - 6 Information', 'JPC - Program - 5 Weather',
       'JPC - Safety - Demolition', 'JPC - Safety - Plant & Equipment',
       'JPC - Safety - Environmental', 'JPC - Quality - Materials',
       'JPC - Safety - Excavations', 'JPC - Quality - Management',
       'JPC - Quality - Damage', 'JPC - Quality - Information',
       'JPC - Safety - Exclusion Zones', 'JPC - Program - 1 Materials',
       'JPC - Safety - Lifting / Slinging',
       'JPC - Safety - Public Interface / Hoar

In [18]:
filtered_forms['Report Forms Group'].unique()

array(['Site Management', 'Subcontractor', 'Quality', 'Safety', nan,
       'Design Team'], dtype=object)

In [19]:
# получим строки с отсутствующими значениями по формам
filtered_forms[filtered_forms.isnull().any(axis=1)]

,Status,OverDue,Type,Report Forms Group,Created,Project,Ref,Name,Report Forms Status
85,Open / Ongoing Works,False,00 Project Management,NaN,2020-09-10,1328,F102497.30,JPC-PM-FRM-001 - Minutes of Meeting,Open
119,Open / Ongoing Works,False,00 Project Management,NaN,2020-09-11,1328,F102497.31,JPC-PM-FRM-001 - Minutes of Meeting,Open
1107,Open / Ongoing Works,False,00 Project Management,NaN,2020-07-06,1328,F102497.23,JPC-PM-FRM-001 - Minutes of Meeting,Open
1109,Open / Ongoing Works,False,00 Project Management,NaN,2020-07-06,1328,F102497.21,JPC-PM-FRM-001 - Minutes of Meeting,Open
4175,Permit to Load Signed Off,False,Safety Forms,Safety,2020-08-18,1329,F92677.386,CM-EHS-FRM-003 Temporary Works / Permit to Load,NaN
4256,Permit to Load Signed Off,False,Safety Forms,Safety,2020-08-05,1329,F92677.357,CM-EHS-FRM-003 Temporary Works / Permit to Load,NaN


In [20]:
filtered_forms['Report Forms Group'] = filtered_forms['Report Forms Group'].replace({np.nan: 'Unknown', 'NaN': 'Unknown'})
filtered_tasks['Priority'] = filtered_tasks['Priority'].replace({np.nan: 'Unknown', float('nan'): 'Unknown', '.': 'Unknown', 'NaN': 'Unknown'})
filtered_tasks['Cause'] = filtered_tasks['Cause'].replace({np.nan: 'Unknown', float('nan'): 'Unknown', '.': 'Unknown', 'NaN': 'Unknown'})

In [21]:
filtered_tasks.sample(8)

,Status,OverDue,Priority,Cause,Created,Project,Ref,Description,Report Status,Task Group
3518,EHS Good Observation,False,Unknown,JPC - Safety - Access,2019-07-12,1328,T84104.39,Very high standard of housekeeping along the p...,Closed,Safety
12042,EHS Good Observation,False,Unknown,JPC - Safety - Management,2020-08-31,1345,T135787.163,Good control of barriers in work zone,Closed,Safety
2441,Closed,False,Behavioural Failure,JPC - Safety - Environmental,2019-11-15,1328,T90896.53,Empty spray cans left on upper floors. Hazardo...,Closed,Safety
6399,JPC Sign Off / Closed,False,Unknown,JPC - Quality - Workmanship,2020-04-27,1330,T124251.20,X812-H-18 - Paint to be touched up where scuffed.,Closed,Quality
7622,Closed,False,Unknown,JPC - Safety - House Keeping,2019-09-29,1330,T97233.17,Chain link fence to be removed to skip,Closed,Safety
3101,Closed,False,System Failure,JPC - Safety - Access,2019-09-03,1328,T76201.253,Poor housekeeping,Closed,Safety
9001,Open / Ongoing Works,False,Unknown,Unknown,2020-02-03,1335,T103672.134,43864,Open,Site Management
1146,EHS Good Observation,False,Unknown,JPC - Safety - Exclusion Zones,2020-03-27,1328,T1.21618936,Townsend street mass climber access. Procedure...,Closed,Safety


In [22]:
filtered_forms.sample(8)

,Status,OverDue,Type,Report Forms Group,Created,Project,Ref,Name,Report Forms Status
3309,Closed JPC Sign Off,False,Quality 01 Structural,Quality,2019-11-25,1328,F84104.534,1328 QM-CL-CS-116 Concrete Cube Record,Closed
3719,Completed and signed off,False,Safety Forms,Safety,2019-06-11,1328,F74906.46,EHS Managment Inspection - Daily,Closed
3931,Closed JPC Sign Off,False,Quality 01 Structural,Quality,2019-08-20,1328,F84104.171,1328 QM-CL-CS-116 Concrete Cube Record,Closed
5143,Closed,False,Site Management,Site Management,2019-09-25,1329,F91214.111,SM-FRM-001 Daily Site Diary,Closed
2149,Closed,False,Quality 00 General,Quality,2020-03-09,1328,F116412.208,QM-FRM-QC-009 Quality Control Record,Closed
8967,Closed JPC Reviewed,False,Site Management,Site Management,2020-07-30,1340,F132455.104,SM-FRM-002 Progress Record,Closed
5946,Closed JPC Reviewed,False,Site Management,Site Management,2020-05-21,1330,F109023.115,SM-FRM-002 Progress Record,Closed
193,Open / Ongoing Works,False,Quality 02 Architectural,Quality,2020-09-08,1328,F131475.88,QM-CL-ARC-204 Void Closure Sign Off (Walls and...,Open


## Подготовим данные для расчетов

In [23]:

# Словарь группировки
group_dict = {
    'Behavioural Failure': 'Failure',
    'Unknown': 'Other',
    'System Failure': 'Failure',
    'High': 'Priority',
    'Medium': 'Priority',
    'Low': 'Priority',
    'Medium (resolve within 5 days)': 'Priority',
    'High (resolve within 48 hours)': 'Priority',
    'Low (resolve within 2 weeks)': 'Priority',
    'Best Practice': 'Best Practice',
    'System Failure - Deviation from RAMS / Manufacturer Instructions': 'Failure',
    '1 Month Look Ahead': 'Planning',
    '2 Week Look Ahead': 'Planning',
    '1 Week Look Ahead': 'Planning'
}

# Применение группировки
filtered_tasks['Priority_Group'] = filtered_tasks['Priority'].map(group_dict)

In [24]:
def extract_word(value):
    parts = value.split('-')
    if len(parts) > 2:
        return parts[1].strip()
    else:
        return 'Unknown'
    return None

filtered_tasks['Cause_'] = filtered_tasks['Cause'].apply(extract_word)

In [25]:
filtered_forms.loc[~filtered_forms['Report Forms Status'].isin(['Closed', 'Open']), 'Report Forms Status'] = 'Closed'

# Данные готовы, экспортируем на визуализацию

In [26]:
filtered_forms.to_csv('df_forms.csv')

In [27]:
filtered_tasks.to_csv('df_tasks.csv')